# Step 2 — Feature Engineering & Model Development
**Goal:** turn the raw data into model-ready features, train **Logistic Regression** and **Random Forest**,
tune them, evaluate with cross-validation, and choose one model with a documented reason.

**The plan in one picture**
```
all data (~920 rows)
 ├── 80% development set ── 5-fold cross-validation ── compare, tune, choose
 └── 20% test set ───────── locked away, used ONCE at the end for the final score
```
Every preprocessing step lives **inside** a scikit-learn `Pipeline`, so it is learned only from training data in each fold
(no leakage) and the exact same steps will run later in the production API.

## 1. Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_validate,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "reports" / "figures"
MODEL_DIR = ROOT / "models"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

SEED = 42   # fixed random seed -> same splits and same results every run (reproducibility)

# ---- Colours: defined once, used by name in every chart ----
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#52514e"
MODEL_COLORS = {"Logistic Regression": BLUE, "Random Forest": ORANGE}
sns.set_theme(style="whitegrid")

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

## 2. Load the data
Same loading and cleaning rules as the EDA notebook: four sources, file checks, `?` → missing, impossible zeros → missing, binary target.

In [ ]:
COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
SOURCES = {
    "cleveland":   "processed.cleveland.data",
    "hungary":     "processed.hungarian.data",
    "switzerland": "processed.switzerland.data",
    "va":          "processed.va.data",
}

frames = []
for source, filename in SOURCES.items():
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"
    raw = pd.read_csv(path, header=None, na_values="?")
    assert raw.shape[1] == len(COLUMNS), f"{filename}: expected 14 columns, found {raw.shape[1]}"
    raw.columns = COLUMNS
    raw["source"] = source
    frames.append(raw)

df = pd.concat(frames, ignore_index=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")
df[["chol", "trestbps"]] = df[["chol", "trestbps"]].replace(0, np.nan)
print(f"{df.shape[0]} rows loaded")

## 3. Define features and split the data
**Feature groups** decide how each column is prepared:

| Group | Columns | Preparation |
|---|---|---|
| Numeric | age, trestbps, chol, thalach, oldpeak, ca | fill missing with **median** → **standard-scale** |
| Categorical | cp, restecg, slope, thal | fill missing with **most frequent** → **one-hot encode** |
| Binary | sex, fbs, exang | fill missing with **most frequent** (already 0/1) |

`ca` (0–3 vessels) is a count, so it is treated as numeric. `source` is **not** a feature.

**Split:** 20% test set held back, stratified on **target + source together** so both sets have the same mix of hospitals and diagnoses.

In [ ]:
NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
CATEGORICAL = ["cp", "restecg", "slope", "thal"]
BINARY = ["sex", "fbs", "exang"]
FEATURES = NUMERIC + CATEGORICAL + BINARY

X = df[FEATURES]
y = df["target"]
strata = df["source"] + "_" + df["target"].astype(str)   # e.g. "cleveland_1"

X_train, X_test, y_train, y_test, strata_train, _ = train_test_split(
    X, y, strata, test_size=0.20, stratify=strata, random_state=SEED)

print(f"Development set: {len(X_train)} rows  |  Test set: {len(X_test)} rows")
print(f"Disease rate  -> development {y_train.mean():.0%}, test {y_test.mean():.0%}")

## 4. Preprocessing pipeline (scaling + encoding)
`add_indicator=True` adds a yes/no column for every feature that had missing values in training — e.g. `missingindicator_ca`.
For `ca`, `thal` and `slope`, *whether* the value is missing carries information (it tells the model which hospital protocol was followed), so the model gets to use it.

The `build_preprocessor` function also lets us test the EDA question: **keep `ca`, `thal`, `slope` (with indicators), or drop them?**

In [ ]:
SPARSE = ["ca", "thal", "slope"]   # heavily missing outside Cleveland (see EDA)

def build_preprocessor(drop_sparse=False):
    num = [c for c in NUMERIC if not (drop_sparse and c in SPARSE)]
    cat = [c for c in CATEGORICAL if not (drop_sparse and c in SPARSE)]

    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    categorical_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent", add_indicator=True)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    binary_steps = SimpleImputer(strategy="most_frequent")

    return ColumnTransformer([
        ("num", numeric_steps, num),
        ("cat", categorical_steps, cat),
        ("bin", binary_steps, BINARY),
    ])

def make_pipeline(model, drop_sparse=False):
    return Pipeline([("prep", build_preprocessor(drop_sparse)), ("model", model)])

# Peek at the prepared features
prep = build_preprocessor().fit(X_train)
feature_names = prep.get_feature_names_out()
print(f"{len(FEATURES)} raw columns  ->  {len(feature_names)} model features")
pd.DataFrame(prep.transform(X_train[:5]), columns=feature_names).round(2)

## 5. Cross-validation setup
5-fold stratified cross-validation on the development set: the data is cut into 5 parts, the model trains on 4 and is scored on the 5th, five times.
We report the **mean ± standard deviation** — the mean is the expected performance, the std shows how stable it is.

| Metric | Question it answers |
|---|---|
| Accuracy | What share of all predictions are correct? |
| Precision | Of patients flagged as diseased, how many really are? |
| **Recall** | Of patients who really have disease, how many did we catch? *(most important here — a missed patient is the costliest error)* |
| **ROC-AUC** | How well does the model rank sick above healthy, across all thresholds? *(main selection metric)* |

*If you see a warning like "The least populated class in y has only N members": that's expected. Switzerland has very few healthy patients,
so that one hospital/diagnosis group can't be spread evenly over 5 folds. It is harmless.*

In [ ]:
SCORING = ["accuracy", "precision", "recall", "roc_auc"]
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_train, strata_train))

def evaluate(pipeline, name):
    """Run 5-fold CV and return one row: mean and std of each metric."""
    scores = cross_validate(pipeline, X_train, y_train, cv=folds, scoring=SCORING, n_jobs=-1)
    row = {"model": name}
    for m in SCORING:
        row[f"{m}_mean"] = scores[f"test_{m}"].mean()
        row[f"{m}_std"] = scores[f"test_{m}"].std()
    return row

def show(rows):
    """Format results as 'mean ± std'."""
    table = pd.DataFrame(rows).set_index("model")
    return pd.DataFrame({m: table[f"{m}_mean"].map("{:.3f}".format) + " ± " +
                            table[f"{m}_std"].map("{:.3f}".format) for m in SCORING})

## 6. Feature-set decision: keep or drop the sparse columns?
A quick, fair test with a default Logistic Regression. Whichever feature set scores better on ROC-AUC is used from here on.

In [ ]:
lr = LogisticRegression(max_iter=1000)
feature_test = [
    evaluate(make_pipeline(lr, drop_sparse=False), "Keep ca/thal/slope + missing indicators"),
    evaluate(make_pipeline(lr, drop_sparse=True),  "Drop ca/thal/slope"),
]
show(feature_test)

In [ ]:
DROP_SPARSE = feature_test[1]["roc_auc_mean"] > feature_test[0]["roc_auc_mean"]
print("Decision:", "DROP ca/thal/slope" if DROP_SPARSE else "KEEP ca/thal/slope with missing indicators")

## 7. Baseline models (default settings)
Both models with scikit-learn defaults. This is the reference point that tuning must beat.

- **Logistic Regression** — linear, fast, interpretable (each feature gets a weight), needs scaled inputs.
- **Random Forest** — many decision trees voting; captures non-linear patterns and interactions, no scaling needed (harmless to keep).

In [ ]:
baseline = [
    evaluate(make_pipeline(LogisticRegression(max_iter=1000), DROP_SPARSE),
             "Logistic Regression (baseline)"),
    evaluate(make_pipeline(RandomForestClassifier(random_state=SEED), DROP_SPARSE),
             "Random Forest (baseline)"),
]
show(baseline)

## 8. Hyperparameter tuning
`GridSearchCV` tries every combination in the grid with the **same 5 folds** and keeps the one with the best ROC-AUC.

**Logistic Regression**
- `C` — regularisation strength. Small C = simpler model (less overfitting), large C = fits training data more closely.
- `l1_ratio` — type of regularisation: `0` (L2) shrinks all weights; `1` (L1) can switch weak features off entirely.

**Random Forest**
- `n_estimators` — number of trees (more = more stable, slower).
- `max_depth` — how deep each tree can grow (limits overfitting).
- `min_samples_leaf` — minimum patients per leaf (larger = smoother decisions).
- `max_features` — features considered per split (adds randomness between trees).

In [ ]:
lr_grid = GridSearchCV(
    make_pipeline(LogisticRegression(max_iter=2000, solver="liblinear"), DROP_SPARSE),
    param_grid={"model__C": [0.01, 0.1, 1, 10], "model__l1_ratio": [0, 1]},
    cv=folds, scoring="roc_auc", n_jobs=-1,
).fit(X_train, y_train)

rf_grid = GridSearchCV(
    make_pipeline(RandomForestClassifier(random_state=SEED), DROP_SPARSE),
    param_grid={
        "model__n_estimators": [200, 400],
        "model__max_depth": [4, 8, None],
        "model__min_samples_leaf": [1, 5, 10],
        "model__max_features": ["sqrt", 0.5],
    },
    cv=folds, scoring="roc_auc", n_jobs=-1,
).fit(X_train, y_train)

for name, grid in [("Logistic Regression", lr_grid), ("Random Forest", rf_grid)]:
    best = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}
    print(f"{name:<20} best ROC-AUC {grid.best_score_:.3f}  with {best}")

In [ ]:
# Top 5 settings tried for each model (documents the tuning process)
def top_results(grid, n=5):
    res = pd.DataFrame(grid.cv_results_)
    params = res["params"].apply(lambda p: ", ".join(f"{k.replace('model__', '')}={v}" for k, v in p.items()))
    return (pd.DataFrame({"params": params, "roc_auc_mean": res["mean_test_score"],
                          "roc_auc_std": res["std_test_score"]})
              .sort_values("roc_auc_mean", ascending=False).head(n).round(3))

display(top_results(lr_grid))
display(top_results(rf_grid))

## 9. Compare baseline vs tuned (cross-validation)

In [ ]:
tuned = [
    evaluate(lr_grid.best_estimator_, "Logistic Regression (tuned)"),
    evaluate(rf_grid.best_estimator_, "Random Forest (tuned)"),
]
results = baseline + tuned
show(results)

In [ ]:
res = pd.DataFrame(tuned).set_index("model")
x = np.arange(len(SCORING))
fig, ax = plt.subplots(figsize=(9, 4))
for i, (name, color) in enumerate(MODEL_COLORS.items()):
    row = res.loc[f"{name} (tuned)"]
    means = [row[f"{m}_mean"] for m in SCORING]
    stds = [row[f"{m}_std"] for m in SCORING]
    ax.bar(x + (i - 0.5) * 0.36, means, width=0.34, yerr=stds, capsize=3,
           color=color, label=name, error_kw={"ecolor": GREY, "elinewidth": 1})
ax.set_xticks(x, ["Accuracy", "Precision", "Recall", "ROC-AUC"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean CV score (± std)")
ax.set_title("Tuned models: 5-fold cross-validation", fontweight="bold")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="x", visible=False)
save(fig, "09_cv_model_comparison")
plt.show()

## 10. Model selection
**Decision rule (written down *before* looking at the test set):**
1. Choose the model with the higher mean CV **ROC-AUC**.
2. If the two are within one standard deviation of each other, the difference is noise → prefer **Logistic Regression**: simpler, faster, interpretable, and easier to explain in a medical setting.

In [ ]:
lr_row = res.loc["Logistic Regression (tuned)"]
rf_row = res.loc["Random Forest (tuned)"]
gap = rf_row["roc_auc_mean"] - lr_row["roc_auc_mean"]
noise = max(lr_row["roc_auc_std"], rf_row["roc_auc_std"])

if gap > noise:
    chosen_name, chosen = "Random Forest", rf_grid.best_estimator_
    reason = f"Random Forest's ROC-AUC is higher by {gap:.3f}, more than one std ({noise:.3f})."
else:
    chosen_name, chosen = "Logistic Regression", lr_grid.best_estimator_
    reason = (f"ROC-AUC difference ({gap:+.3f}) is within one std ({noise:.3f}), "
              "so the simpler, interpretable model wins.")
print(f"Selected model: {chosen_name}\nReason: {reason}")

## 11. Final evaluation on the held-out test set
The test set is used **once**, now, for both tuned models. This is the honest estimate of performance on new patients.

In [ ]:
final = {"Logistic Regression": lr_grid.best_estimator_, "Random Forest": rf_grid.best_estimator_}
rows = []
for name, model in final.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_test, pred),
                 "precision": precision_score(y_test, pred),
                 "recall": recall_score(y_test, pred),
                 "roc_auc": roc_auc_score(y_test, proba)})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, model in final.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, color=MODEL_COLORS[name], linewidth=2,
                 label=f"{name} (AUC {roc_auc_score(y_test, proba):.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color=GREY, linewidth=1)
axes[0].set_xlabel("False positive rate (healthy patients flagged)")
axes[0].set_ylabel("True positive rate (recall)")
axes[0].legend(frameon=False, loc="lower right")
axes[0].set_title("ROC curves (test set)", fontweight="bold")

for ax, (name, model) in zip(axes[1:], final.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, cmap="Blues",
                                          display_labels=["No disease", "Disease"], colorbar=False)
    ax.set_title(f"{name}: confusion matrix", fontweight="bold")
    ax.grid(False)
fig.tight_layout()
save(fig, "10_test_roc_confusion")
plt.show()

**Reading the confusion matrix:** the bottom-left cell is **false negatives** — sick patients the model called healthy.
That is the number we most want to keep small.

## 12. What drives the predictions?
- **Logistic Regression:** coefficient size and sign (positive = raises disease risk).
- **Random Forest:** how much each feature reduces impurity across all trees.

In [ ]:
lr_best = lr_grid.best_estimator_
coefs = pd.Series(lr_best.named_steps["model"].coef_[0],
                  index=lr_best.named_steps["prep"].get_feature_names_out())
rf_best = rf_grid.best_estimator_
importances = pd.Series(rf_best.named_steps["model"].feature_importances_,
                        index=rf_best.named_steps["prep"].get_feature_names_out())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
top_lr = coefs.reindex(coefs.abs().sort_values().index)[-12:]
axes[0].barh(top_lr.index, top_lr.values, color=[ORANGE if v > 0 else BLUE for v in top_lr.values])
axes[0].axvline(0, color=GREY, linewidth=1)
axes[0].set_title("Logistic Regression: top coefficients\n(orange raises risk, blue lowers it)", fontweight="bold")

top_rf = importances.sort_values()[-12:]
axes[1].barh(top_rf.index, top_rf.values, color=ORANGE)
axes[1].set_title("Random Forest: top feature importances", fontweight="bold")
for ax in axes:
    ax.grid(axis="y", visible=False)
fig.tight_layout()
save(fig, "11_feature_importance")
plt.show()

## 13. Save the selected model
The **whole pipeline** (imputation + scaling + encoding + model) is saved as one file. The API in Step 4 will load this file and
can be given raw patient values directly — no separate preprocessing code to keep in sync.

*(In Step 3, MLflow will take over tracking and storing models; this file is the simple version.)*

In [ ]:
chosen.fit(X_train, y_train)          # already fitted by GridSearchCV; explicit for clarity
model_path = MODEL_DIR / "heart_disease_model.joblib"
joblib.dump(chosen, model_path)
print(f"Saved {chosen_name} pipeline -> {model_path.relative_to(ROOT)}")

# Sanity check: reload and predict one patient from raw values
reloaded = joblib.load(model_path)
sample = X_test.iloc[[0]]
print("Raw input:", sample.to_dict("records")[0])
print(f"Predicted disease probability: {reloaded.predict_proba(sample)[0, 1]:.2f}")

## 14. Model selection & tuning summary
*Fill this in from **your** outputs — it becomes the "Model Development" section of your report.*

**Features**
- Final feature set (kept or dropped `ca`/`thal`/`slope`, and the CV evidence): …
- Preprocessing: median / most-frequent imputation with missing indicators, standard scaling, one-hot encoding — all inside one Pipeline.

**Baseline → tuned** (CV ROC-AUC, mean ± std)
- Logistic Regression: … → … (best params: …)
- Random Forest: … → … (best params: …)
- Did tuning help meaningfully, or was the gain within noise? …

**Selected model:** … **Reason:** …

**Test-set results** (selected model): accuracy …, precision …, recall …, ROC-AUC …
- Do the test scores agree with CV (no big drop = no overfitting)? …
- False negatives on the test set: …

**Most influential features:** … (do they match the EDA findings?)

**Limitations:** small dataset (~920 rows), historical data (1980s), heavy missingness outside Cleveland, hospitals differ in disease rate. Not for clinical use.